## 4.1 Setup and output paths

This section loads the libraries used for splitting, preprocessing, modeling, and reporting, points the notebook to the cleaned dataset, and prepares the output folders so subsequent sections can write CSVs and figures without further setup.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, f_oneway

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    PredefinedSplit,
    StratifiedKFold,
    StratifiedShuffleSplit,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", context="talk")

RANDOM_STATE = 42

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATASET_PATH = ROOT / "data" / "processed" / "dataset_final.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
FOLDS_DIR = PROCESSED_DIR / "ml_stratified_5folds"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
for output_directory in (FIGURES_DIR, TABLES_DIR, FOLDS_DIR):
    output_directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Dataset path: {DATASET_PATH.relative_to(ROOT)}")
print(f"Folds directory: {FOLDS_DIR.relative_to(ROOT)}")
print(f"Tables directory: {TABLES_DIR.relative_to(ROOT)}")
print(f"Figures directory: {FIGURES_DIR.relative_to(ROOT)}")

Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Dataset path: data/processed/dataset_final.csv
Folds directory: data/processed/ml_stratified_5folds
Tables directory: outputs/tables
Figures directory: outputs/figures


In [ ]:
assert DATASET_PATH.exists(), "Run data_collection.ipynb first so data/processed/dataset_final.csv exists."

date_columns = ["service_date", "scheduled_arrival", "scheduled_arrival_date", "weather_time"]
df = pd.read_csv(DATASET_PATH, parse_dates=date_columns)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 90,818
Columns: 21


,service_date,season,scheduled_arrival,scheduled_arrival_date,delay_minutes,delayed,scheduled_arrival_hour,scheduled_arrival_weekday,scheduled_arrival_weekday_name,scheduled_arrival_month,...,stop_name,train_type,line_name,operator,weather_time,temperature_2m,precipitation,snowfall,wind_speed_10m,weather_matched
0,2025-01-01,Winter,2025-01-01 01:25:00,2025-01-01,-0.200000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
1,2025-01-01,Winter,2025-01-01 01:27:00,2025-01-01,1.000000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
2,2025-01-01,Winter,2025-01-01 01:34:00,2025-01-01,1.300000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
3,2025-01-01,Winter,2025-01-01 01:35:00,2025-01-01,-0.650000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
4,2025-01-01,Winter,2025-01-01 01:52:00,2025-01-01,0.183333,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
